In [ ]:
# UK House Price Prediction
# Author: Cosmin Tutea
# Dataset: UK House Price Register 2015-2024 (Kaggle)
# Description: End-to-end machine learning project to predict UK house prices

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import joblib
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

In [ ]:
# Load the dataset
# File picker opens automatically if CSV is not found in current folder

default_file = Path("../data/UK_House_Price_Prediction_dataset_2015_to_2024.csv")

if default_file.exists():
    csv_path = str(default_file)
    print(f"Dataset found: {csv_path}")
else:
    print("Dataset not found. Opening file picker...")
    from tkinter import Tk
    from tkinter.filedialog import askopenfilename

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    csv_path = askopenfilename(
        title="Select UK House Price CSV File",
        filetypes=[("CSV files", "*.csv")]
    )
    root.destroy()

df = pd.read_csv(csv_path)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Explore the dataset

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
df.describe()

In [ ]:
# Visualise the dataset

# Price distribution
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of UK House Prices")
plt.xlabel("Price (£)")
plt.ylabel("Frequency")
plt.show()

# Price by property type
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="property_type", y="price", estimator=np.mean)
plt.title("Average House Price by Property Type")
plt.xlabel("Property Type")
plt.ylabel("Average Price (£)")
plt.show()

# Price trend by year
plt.figure(figsize=(10, 5))
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df.groupby("year")["price"].median().plot(marker="o")
plt.title("Median House Price by Year")
plt.xlabel("Year")
plt.ylabel("Median Price (£)")
plt.show()

In [ ]:
# Feature engineering

# Extract date features
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

# Extract postcode area
df["postcode_area"] = df["postcode"].astype(str).str.split().str[0]

# Remove outliers - top 1% most expensive properties
upper_limit = df["price"].quantile(0.99)
print(f"99th percentile price: £{upper_limit:,.2f}")
df = df[df["price"] <= upper_limit]
print(f"Shape after outlier removal: {df.shape}")

# Drop columns not needed for modelling
df = df.drop(columns=["date", "postcode", "street", "locality"])
df = df.dropna()

print("Feature engineering complete.")
df.head()

In [ ]:
# Prepare features and target

X = df.drop(columns=["price"])
y = df["price"]

categorical_features = ["property_type", "new_build", "freehold", "town", "district", "county", "postcode_area"]
numerical_features = ["year", "month", "quarter"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

In [ ]:
# Train and evaluate models

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_split=5, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42, objective="reg:squarederror")
}

results = {}
best_model = None
best_model_name = None
best_rmse = float("inf")

for model_name, model in models.items():
    print(f"Training: {model_name}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results[model_name] = {"RMSE": rmse, "MAE": mae, "R2": r2}

    print(f"RMSE: £{rmse:,.2f} | MAE: £{mae:,.2f} | R2: {r2:.4f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_model = pipeline
        best_model_name = model_name

print(f"\nBest model: {best_model_name} with RMSE £{best_rmse:,.2f}")

In [ ]:
# Cross-validation

print("Cross-Validation Results (5-fold):")
print("=" * 60)

for model_name, model in models.items():
    pipeline_cv = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    cv_scores = cross_val_score(
        pipeline_cv, X, y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    cv_rmse = -cv_scores
    print(f"{model_name}:")
    print(f"  Mean RMSE: £{cv_rmse.mean():,.2f}")
    print(f"  Std RMSE:  £{cv_rmse.std():,.2f}")

In [ ]:
# Visualise model performance

results_df = pd.DataFrame(results).T.sort_values(by="RMSE")

print("Model Performance Summary:")
print(results_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=results_df.index, y=results_df["RMSE"])
plt.title("Model Comparison using RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE (£)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Save the best model and feature options

joblib.dump(best_model, "../models/best_house_price_model.pkl")

feature_options = {
    "property_type": sorted(df["property_type"].unique().tolist()),
    "new_build": sorted(df["new_build"].unique().tolist()),
    "freehold": sorted(df["freehold"].unique().tolist()),
    "town": sorted(df["town"].unique().tolist()),
    "district": sorted(df["district"].unique().tolist()),
    "county": sorted(df["county"].unique().tolist()),
    "postcode_area": sorted(df["postcode_area"].unique().tolist()),
    "year": {"min": int(df["year"].min()), "max": int(df["year"].max()), "median": int(df["year"].median())},
    "month": {"min": int(df["month"].min()), "max": int(df["month"].max()), "median": int(df["month"].median())},
    "quarter": {"min": int(df["quarter"].min()), "max": int(df["quarter"].max()), "median": int(df["quarter"].median())}
}

joblib.dump(feature_options, "../models/feature_options.pkl")

print(f"Best model saved: {best_model_name}")
print("Feature options saved.")